# STAR — Kaggle V6: COMBO train (pair-batch + text-LoRA THAT + ViTPose) → TEST ngay

Lay cong thuc thang **v3c (pair-batch + ViTPose) = 0.7485** roi cong them **text-LoRA**, **epoch 10**.
Train xong **chay LUON tren old-test THAT** (1.978 anh, POSE-OFF) — khong dung o VAL-B.

🔴 **V6 la lan DAU text-LoRA chay THAT.** v3a/v3b truoc day bi bug bool (`lora_freeze_text=false` → chuoi "false"
truthy → text VAN DONG BANG) nen "text-LoRA +0.01" la KET LUAN SAI — chua tung test. Bug da fix → gio text MO that
(60 lora-layer, 3.27M trainable, **14.0G**) vs frozen cu (48 layer, 2.98M, 13.5G).

🔴 **FIX OOM (lan truoc chet o step 200):** nguyen nhan = diagnostic grad-norm (3x backward retain_graph) cong them tren
nen 14.0G → vuot 14.56G usable. Da sua: **batch 16** (~12G, KHONG 20) + **`grad_norm_every=0`** (tat diagnostic) +
**`expandable_segments`** (chong phan manh). Bao dam fit.

⚠️ **TEST chay POSE-OFF** (khong YOLO/ViTPose, tranh xung dot env). Model train POSE-ON nhung test pose-OFF → `stage1`
lech train/eval → nhin **`rerank`** (khong dung pose) cho cong bang.

**VAL-B** (split seed-42 nhu v3) chi de chon best.pth; **ket qua chinh = old-test THAT o cuoi.**

**Add Input (2):** `star-10k-hard` (tar.zst + xvlm_16m_base.th) · `star-oldtest` (attr.json + test/). **GPU T4 · Internet ON.**
**Thoi gian (sua lai):** text-LoRA that **+66%/step** (2.63 vs 1.59 s/step) → batch16 × 10 epoch ≈ **~6.5–7.5h**
(co early-stop, co the cat som). Muon nhanh: ha `epochs=6` o cell 5 (~4.5h). Best.pth: `/kaggle/working/v6/best.pth`.

In [ ]:
# [1/9] GPU + clone/pull repo (can ban MOI: PairBatchSampler + FIX bool override + pose-in-pipeline + yolo)
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "KHONG THAY GPU — bat Settings > Accelerator = GPU T4!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
assert "PairBatchSampler" in pathlib.Path("src/star/data/sampler.py").read_text()
assert 'low == "true"' in pathlib.Path("src/star/config.py").read_text(), "thieu FIX bool — pull lai!"
assert "keypoints" in pathlib.Path("src/star/inference/pipeline.py").read_text()
assert pathlib.Path("scripts/extract_pose_yolo.py").exists()
print("repo OK:", os.getcwd())

In [ ]:
# [2/9] Env pinned + X-VLM + 4 patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/9] Tim dataset train (tar.zst) + old-test + base ckpt
import glob, os, pathlib, shutil
def find_all(p):
    return sorted(set(glob.glob(f"/kaggle/input/*/{p}") + glob.glob(f"/kaggle/input/*/**/{p}", recursive=True)))
def find_one(p):
    h = find_all(p); return h[0] if h else None

CKPT  = find_one("xvlm_16m_base.th")
JSONL = find_one("train_10k_hard.jsonl")
ARCH  = find_one("train_10k_hard_data.tar.zst")
if JSONL:
    DATA_ROOT = str(pathlib.Path(JSONL).parents[2])
elif ARCH:
    marker = "/kaggle/working/extract"
    got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    if not got:
        os.makedirs(marker, exist_ok=True); print("extracting 1.6GB (~2-4')...")
        if shutil.which("zstd"):
            os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {marker}")
        else:
            import zstandard, tarfile
            with open(ARCH, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
                with tarfile.open(fileobj=zr, mode="r|") as tf:
                    tf.extractall(marker)
        got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    assert got, "giai nen that bai"; DATA_ROOT = str(pathlib.Path(got[0]).parents[2])
else:
    raise FileNotFoundError("Khong thay data train! Add Input star-10k-hard.")
if not CKPT:
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(CKPT).stat().st_size > 8e8

# old-test (de TEST cuoi) — neu chua add se canh bao o cell 8
ATTR = find_one("attr.json")
OLD_ROOT = None
if ATTR:
    OLD_ROOT = str(pathlib.Path(ATTR).parent)
    pr = find_all("test/0.jpg") or find_all("0.jpg")
    if pr and not pathlib.Path(OLD_ROOT, "test/0.jpg").exists():
        OLD_ROOT = str(pathlib.Path(pr[0]).parents[1])
print("DATA_ROOT =", DATA_ROOT, "| CKPT =", CKPT)
print("OLD_ROOT  =", OLD_ROOT or "(CHUA add star-oldtest -> bo qua test cuoi)")

In [ ]:
# [4/9] Build manifest train (GIONG HET v3: keypoints 100% + bbox + pair_image_id + split video seed-42)
import json, pathlib
import numpy as np, pandas as pd
root = pathlib.Path(DATA_ROOT)
anchors, hard_rows, anchor_ids = [], {}, set()
def bucket_of(p):
    return "wentwrong" if "/wentwrong/" in p else ("full" if "/full/" in p else "goal")
for line in open(root / "data/subsets/train_10k_hard.jsonl", encoding="utf-8"):
    r = json.loads(line); anchor_ids.add(r["image_id"])
    anchors.append(dict(image_path=r["image_webp"], caption=r["caption"],
        sequence_id=f'v{r["video_id"]}_{r["bucket"]}', scene=f'v{r["video_id"]}',
        action=str(r.get("label", "unk")), video_id=r["video_id"], image_id=r["image_id"],
        pair_image_id=r.get("hard_i_id")))
    hid = r.get("hard_i_id")
    if hid and hid not in hard_rows:
        hard_rows[hid] = dict(image_path=r["hard_image_webp"], caption=r["hard_c"],
            sequence_id=f'v{r["video_id"]}_{bucket_of(r["hard_image_webp"])}', scene=f'v{r["video_id"]}',
            action="hard_pair", video_id=r["video_id"], image_id=hid, pair_image_id=None)
df = pd.DataFrame(anchors + [v for k, v in hard_rows.items() if k not in anchor_ids])
pose = json.load(open(root / "data/subsets/prepared/train_10k_hard_vitpose.json", encoding="utf-8"))["items"]
def kpts_of(iid):
    it = pose.get(iid)
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return None
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else None
def bbox_of(iid):
    it = pose.get(iid); b = it.get("primary_bbox_norm_xyxy") if it else None
    if not b:
        return None
    x1, y1, x2, y2 = b
    return [x1, y1, max(x2 - x1, 1e-3), max(y2 - y1, 1e-3)]
df["keypoints"] = df.image_id.map(kpts_of); df["bbox"] = df.image_id.map(bbox_of)
rng = np.random.default_rng(42)
vids = df.video_id.unique().copy(); rng.shuffle(vids)
counts = df.groupby("video_id").size(); vq, vd, acc = set(), set(), 0
it = iter(vids)
for v in it:
    vq.add(v); acc += counts[v]
    if acc >= 600: break
acc = 0
for v in it:
    vd.add(v); acc += counts[v]
    if acc >= 900: break
df["split"] = np.where(df.video_id.isin(vq | vd), "valb", "train")
df.loc[df.video_id.isin(vd), "caption"] = ""
MANIFEST = "/kaggle/working/manifest_10k_hard.parquet"; df.to_parquet(MANIFEST, index=False)
leak = len(set(df[df.split == "train"].video_id) & set(df[df.split == "valb"].video_id))
print(f"rows={len(df)} train={(df.split=='train').sum()} "
      f"valb_q={((df.split=='valb') & (df.caption!='')).sum()} kpts={df.keypoints.notna().mean():.0%} leakage={leak}")
assert leak == 0 and df.keypoints.notna().mean() > 0.99

In [ ]:
# [5/9] ===== CAU HINH V6 ===== (pair + text-LoRA THAT + ViTPose + LHP)
# batch 16 (KHONG 20): text-LoRA mo that -> 14.0G@batch20 + diagnostic = OOM. batch16 ~12G, fit chac.
# grad_norm_every=0: TAT diagnostic 3x-backward (chinh la cho OOM step 200). expandable_segments: chong phan manh.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DATA = f"data.manifest={MANIFEST} data.image_root={DATA_ROOT} model.checkpoint={CKPT}"
LHP = "true"     # v3c vo tinh bat LHP; giu de khong tut. 'false' = tat that (bug da fix).
V6 = (f"data.group_by=pair model.lora_freeze_text=false model.pose_enabled=true "
      f"data.lhp_enabled={LHP} loss.lambda_smooth_ap=0.2 optim.lr_lora=2e-4 "
      f"optim.epochs=10 train.early_stop_patience=10 train.eval_every_epochs=0.5 "
      f"train.batch_size=16 train.grad_accum=2 train.grad_norm_every=0 train.out_dir=/kaggle/working/v6")
print("V6:", V6)

In [ ]:
# [6/9] GATE overfit-one-batch (probe 'vram='). FIX da apply: batch16 + grad_norm off + expandable.
gate = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --overfit-one-batch --set {DATA} {V6}"
print(gate)
!{gate}
print("Neu VAN OOM (hiem): ha batch_size=12 trong V6 (cell 5) roi chay lai tu cell nay.")

In [ ]:
# [7/9] TRAIN V6 (~3.5-4.5h) -> best.pth (VAL-B chi de chon checkpoint)
import pathlib, time, torch
cmd = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --set {DATA} {V6}"
print(cmd)
t0 = time.time()
!{cmd}
mins = round((time.time() - t0) / 60, 1)
BEST = "/kaggle/working/v6/best.pth"
assert pathlib.Path(BEST).exists(), "train fail — xem log (OOM? ha batch_size=16)"
raw = torch.load(BEST, map_location="cpu", weights_only=False)
print(f"V6 train xong ({mins}'): VAL-B mAP={raw.get('best_metric'):.4f} (moc v3c=0.7485) | report={(raw.get('extra') or {}).get('report')}")
del raw

In [ ]:
# [8/9] TEST manifest tren OLD-TEST THAT — gallery-1978, KHONG distractor, KHONG pose (pose-OFF)
import json
import pandas as pd
assert OLD_ROOT, "Chua Add Input star-oldtest -> khong test duoc. Add roi chay lai tu cell 3."
rows = [json.loads(l) for l in open(ATTR, encoding="utf-8")]
def cap_of(r, mode):
    return (r.get("source_caption") or r["caption"])[0] if mode == "human" else r["caption"][0]
TEST_MANIFESTS = {}
for mode in ["human", "vlm"]:
    out = [dict(image_path=r["image"], caption=cap_of(r, mode), split="valb",
                sequence_id=f'o{r["image_id"]}', scene=f'o{r["image_id"]}', action="q",
                image_id=str(r["image_id"]), bbox=None, keypoints=None) for r in rows]   # keypoints=None -> pose-OFF
    mp = f"/kaggle/working/test_{mode}.parquet"; pd.DataFrame(out).to_parquet(mp, index=False); TEST_MANIFESTS[mode] = mp
print(f"test manifests OK | gallery-1978 (khong distractor, POSE-OFF) | modes={list(TEST_MANIFESTS)}")

In [ ]:
# [9/9] CHAY model V6 tren old-test THAT (POSE-OFF) -> mAP/R@1/R@5/R@10 (3 tang, 2 che do caption)
import sys, time, json, pathlib, torch
sys.path.insert(0, "src")
from star.config import load_config, _merge
from star.data import PABDataset
from star.inference import run_pipeline
from star.models import STARModel

raw = torch.load(BEST, map_location="cpu", weights_only=False)
cfg = load_config("configs/star_v3_10k_kaggle.yaml")
_merge(cfg.model, ((raw.get("extra") or {}).get("cfg") or {}).get("model", {}))
cfg.model.checkpoint = CKPT
model = STARModel(cfg).to("cuda").eval()
model.load_state_dict(raw["model"], strict=False); del raw
print(f"loaded V6 best.pth | pose branch={'co' if model.pose is not None else 'khong'} "
      f"| TEST POSE-OFF (manifest khong keypoints -> pipeline bo qua pose)")
if model.pose is not None:
    print("  ⚠️  model POSE-ON nhung test POSE-OFF -> stage1 lech train/eval; xem 'rerank' de cong bang")

RES = []
for mode, mp in TEST_MANIFESTS.items():
    ds = PABDataset(mp, OLD_ROOT, model.backbone.tokenizer, split="valb",
                    image_size=cfg.data.image_size, max_token=cfg.data.max_token, train=False)
    t0 = time.time(); res = run_pipeline(model, ds, "cuda", topk=100, batch_size=64, num_workers=2)
    for stage in ("stage1", "rerank", "gale_shapley"):
        RES.append(dict(mode=mode, stage=stage, **{k: round(v, 4) for k, v in res[stage].items()}))
    print(f"  [{mode:5s}] stage1={res['stage1']['mAP']:.4f} rerank={res['rerank']['mAP']:.4f} "
          f"GS={res['gale_shapley']['mAP']:.4f} ({(time.time()-t0)/60:.1f}')")
import pandas as pd
print("\n===== V6 tren OLD-TEST THAT (gallery 1978, khong distractor, POSE-OFF) =====")
print(pd.DataFrame(RES).to_string(index=False))
pathlib.Path("/kaggle/working/v6_test_oldtest.json").write_text(json.dumps(RES, indent=2))
print("\nLuu: /kaggle/working/v6/best.pth + /kaggle/working/v6_test_oldtest.json")

## Doc ket qua
- **VAL-B mAP** (cell 7): so voi v3c=0.7485 — combo co cai thien tren SYNTHETIC khong.
- **OLD-TEST THAT** (cell 9, **POSE-OFF**): con so that. So `rerank` voi moc v4 cu (~0.19 pure-1978).
  - Vi model POSE-ON ma test pose-OFF → `stage1` bi phat (thieu nhanh pose); `rerank` khong dung pose nen cong bang.
  - `human` (caption ngan) vs `vlm` (chi tiet): do gap ngon ngu.
- **Tai ve:** Save Version → Output → `/kaggle/working/v6/best.pth` + `v6_test_oldtest.json`.
- **Muon test CO pose** (khop train, het lech stage1): chay rieng notebook V5 voi ViTPose/YOLO json — KHONG cai trong V6 nay.
- **Nhac luat:** test (masked) nam nay KHONG dung lam validation/distractor.